### Load the Model

In [ ]:
# Library that converts sentences into embedding vectors
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


W1113 01:29:36.059000 5789 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Encode Sentences

In [2]:
passages = [
    "Total net sales increased 8% compared to the prior year, driven by iPhone and Services.",
    "The company faces intense competition in the smartphone market.",
    "We returned $90 billion to shareholders through dividends and share repurchases.",
    "Our headquarters is located in Cupertino, California.",
]

embeddings = model.encode(passages)		# Converts each sentence into one vector
print(embeddings.shape)

(4, 384)


### Semantic Search

In [7]:
query = "How much did revenue grow?"
query_embedding = model.encode(query)

# Cosine similarity: measures how closely two vectors point in the same direction
scores = util.cos_sim(query_embedding, embeddings)[0]

for passage, score in sorted(zip(passages, scores), key=lambda x: x[1], reverse=True):
	print(f"{score:.3f}   {passage}")

0.507   Total net sales increased 8% compared to the prior year, driven by iPhone and Services.
0.377   We returned $90 billion to shareholders through dividends and share repurchases.
0.191   The company faces intense competition in the smartphone market.
0.102   Our headquarters is located in Cupertino, California.


In [8]:
queries = [
	"How much did revenue grow?",
	"Who are the competitors?",
	"Where is the company based?",
	"What was the operating margin?",	# no answer in passages
]

passage_embeddings = model.encode(passages)

for query in queries:
	scores = util.cos_sim(model.encode(query), passage_embeddings)[0]
	ranked = sorted(zip(passages, scores), key=lambda x: x[1], reverse=True)
	print(f"\nQ: {query}")
	for passage, score in ranked[:2]:
		print(f"   {score:.3f}   {passage[:60]}...")


Q: How much did revenue grow?
   0.507   Total net sales increased 8% compared to the prior year, dri...
   0.377   We returned $90 billion to shareholders through dividends an...

Q: Who are the competitors?
   0.551   The company faces intense competition in the smartphone mark...
   0.183   Our headquarters is located in Cupertino, California....

Q: Where is the company based?
   0.585   Our headquarters is located in Cupertino, California....
   0.306   The company faces intense competition in the smartphone mark...

Q: What was the operating margin?
   0.362   Total net sales increased 8% compared to the prior year, dri...
   0.309   We returned $90 billion to shareholders through dividends an...


**Result:** All three answerable questions retrieved the correct passage at rank 1, including "revenue" matching "net sales" with no shared keyword.

**No-answer question:** "What was the operating margin?" still returned a top result (the net sales passage). Two warning signs: a lower top score (0.362 vs. 0.51-0.59) and a small gap between rank 1 and 2 (0.05 vs. 0.13-0.37).

**Takeaway:** Retrieval always returns something, even when no answer exists. A topically related passage can lead the LLM to hallucinate. The evaluation set must include unanswerable questions.